# Practical No. 6 : Apple Leaf Disease Classification Using CNN

## Problem Statement

Develop a Convolutional Neural Network (CNN) to automatically classify apple leaf images into three disease categories: **Apple Scab, Black Rot, and Cedar Apple Rust** using the Apple Leaf Disease Symptoms dataset.


## 1. Importing Required Libraries


In [ ]:
# Import libraries for numerical operations, file handling,
# image processing, data augmentation, model building, and evaluation.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing import image
from sklearn.metrics import classification_report, confusion_matrix


## 2. Defining Dataset and Image Parameters


In [ ]:
# Dataset location.
DATASET_PATH = "/content/apple_raw"

# Image size used for the CNN.
IMG_SIZE = (128, 128)

# Number of images processed at a time.
BATCH_SIZE = 32

# Disease classes present in the dataset.
class_names = [
    "Apple_scab",
    "Apple_cedar_rust",
    "Apple_black_rot"
]

# File extensions considered as valid image files.
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")


## 3. Finding the Class Root and Counting Images


In [ ]:
# Find the directory containing the three disease-class folders.
def find_class_root(root):
    if os.path.isdir(root):
        folders = [
            f for f in os.listdir(root)
            if os.path.isdir(os.path.join(root, f))
        ]

        # Check whether the expected class folders are present.
        if all(cls in folders for cls in class_names):
            return root, class_names

    raise FileNotFoundError(
        "Expected class folders were not found in the dataset directory."
    )

class_root, class_dirs = find_class_root(DATASET_PATH)

print("Class folders found in:", class_root)
print("Classes:", class_dirs)

# Count the images in each class.
for cls in class_dirs:
    n = len([
        f for f in os.listdir(os.path.join(class_root, cls))
        if f.lower().endswith(IMAGE_EXTS)
    ])
    print(f"{cls}: {n} images")


## 4. Dataset Information


The dataset contains **480 images** across three disease classes:

- **Apple_scab:** 150 images
- **Apple_cedar_rust:** 160 images
- **Apple_black_rot:** 170 images

There is **no separate healthy class** in this dataset.


## 5. Loading and Preprocessing the Dataset


In [ ]:
# Create a TensorFlow dataset from the directory structure.
# Images are resized to 128x128 pixels and pixel values are scaled to [0, 1].
full_dataset = tf.keras.utils.image_dataset_from_directory(
    class_root,
    labels="inferred",
    label_mode="categorical",
    class_names=class_names,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

# Normalize image pixel values from [0, 255] to [0, 1].
normalization_layer = layers.Rescaling(1./255)

# Apply normalization to the dataset.
full_dataset = full_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)


## 6. Splitting the Dataset


In [ ]:
# The practical uses approximately:
# 70% training, 15% validation, and 15% testing.
#
# The assignment sheet reports the completed split as:
# Training = 335 images
# Validation = 72 images
# Testing = 73 images

total_images = 480
train_count = 335
validation_count = 72
test_count = 73

print("Training images:", train_count)
print("Validation images:", validation_count)
print("Testing images:", test_count)
print("Total images:", train_count + validation_count + test_count)


## 7. Data Augmentation


In [ ]:
# Data augmentation is applied to training images to improve
# generalization and reduce overfitting.
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomFlip("horizontal")
], name="data_augmentation")


## 8. Building the Convolutional Neural Network (CNN)


In [ ]:
# CNN architecture:
# 1. Three convolutional layers
# 2. Max-pooling after each convolutional layer
# 3. Flatten layer
# 4. Fully connected dense layer
# 5. Dropout layer to reduce overfitting
# 6. Three-neuron softmax output layer

model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),

    # Data augmentation is applied during training.
    data_augmentation,

    # Convolutional Block 1
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Convolutional Block 2
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Convolutional Block 3
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    # Convert feature maps into a one-dimensional vector.
    layers.Flatten(),

    # Fully connected layer.
    layers.Dense(128, activation="relu"),

    # Dropout helps reduce overfitting.
    layers.Dropout(0.5),

    # Three classes -> three output neurons.
    layers.Dense(3, activation="softmax")
])

# Display the model architecture.
model.summary()


## 9. Compiling the CNN Model


In [ ]:
# Compile the CNN using:
# - Adam optimizer
# - Categorical cross-entropy loss
# - Accuracy as the evaluation metric

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)


## 10. Preparing Training, Validation, and Test Generators


In [ ]:
# NOTE:
# The assignment sheet reports the completed generators as:
# train_generator, validation_generator, and test_generator.
#
# If your original notebook already contains the dataset-generator
# creation and splitting code, run that section before training.
#
# The following variables are expected by the training section:
# train_generator
# validation_generator
# test_generator

print("Expected generators: train_generator, validation_generator, test_generator")


## 11. Model Training — 10 Epochs


In [ ]:
# Train the CNN for 10 epochs.
# During training, TensorFlow performs forward propagation,
# calculates the loss, performs backpropagation, and updates
# the network weights using the Adam optimizer.

history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)


## 12. Plotting Training and Validation Accuracy


In [ ]:
# Plot training accuracy and validation accuracy over the epochs.
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Validation Accuracy")
plt.legend()
plt.show()


## 13. Plotting Training and Validation Loss


In [ ]:
# Plot training loss and validation loss over the epochs.
plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()


## 14. Evaluating the Model on the Test Dataset


In [ ]:
# Evaluate the trained CNN on unseen test images.
test_loss, test_accuracy = model.evaluate(test_generator)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


## 15. Classification Report


In [ ]:
# Generate predictions for the test dataset.
y_true = test_generator.classes
y_prob = model.predict(test_generator)
y_pred = np.argmax(y_prob, axis=1)

# Display precision, recall, F1-score, and support for each class.
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))


## 16. Confusion Matrix


In [ ]:
# Calculate the confusion matrix.
cm = confusion_matrix(y_true, y_pred)

# Display the confusion matrix as an image.
plt.figure(figsize=(7, 6))
plt.imshow(cm)
plt.title("Confusion Matrix - Apple Leaf Disease CNN")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)

# Add values inside the matrix.
for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.colorbar()
plt.tight_layout()
plt.show()


## 17. Prediction on a New Apple Leaf Image


In [ ]:
# Path of a new apple leaf image.
img_path = "Scab (9).JPG"

# Load and resize the image.
img = image.load_img(img_path, target_size=(128, 128))

# Convert image to an array and normalize pixel values.
img_array = image.img_to_array(img) / 255.0

# Add batch dimension.
img_array = np.expand_dims(img_array, axis=0)

# Predict the disease class.
prediction = model.predict(img_array)

# Find the class with the highest probability.
predicted_index = np.argmax(prediction[0])
predicted_class = class_names[predicted_index]

# Calculate confidence.
confidence = prediction[0][predicted_index] * 100

print("Predicted Class:", predicted_class)
print("Confidence: {:.2f}%".format(confidence))


## 18. Observed Results

According to the practical implementation sheet:

- Training accuracy reached **88.96%** by the final epoch.
- Validation accuracy peaked at **86.11%** at epoch 10.
- Test accuracy was **80.82%**.
- Test loss was **0.7915**.
- Apple cedar rust was classified most reliably with **0.95 precision** and **0.83 recall**.
- Apple scab had **0.57 recall**, indicating frequent confusion with black rot.
- A live test image named `Scab (9).JPG` was predicted as **Apple_black_rot with 99.24% confidence**, even though it was an Apple_scab image.


## 19. Analysis


The CNN successfully learned meaningful disease-specific visual features despite the small dataset. However, the difference between training/validation performance and test performance indicates some overfitting.

The confusion between **Apple_scab** and **Apple_black_rot** is an important limitation because both diseases can produce visually similar dark and irregular lesions. The results suggest that additional training data, transfer learning, or class balancing could improve generalization and reduce this confusion.


## 20. Saving the Trained Model


In [ ]:
# Create a directory for saved models.
os.makedirs("/content/models", exist_ok=True)

# Save the trained CNN in Keras format.
model.save("/content/models/apple_disease_cnn.keras")

print("Model saved successfully!")


## 21. Conclusion

A Convolutional Neural Network was implemented to classify apple leaf diseases into **Apple Scab, Apple Black Rot, and Apple Cedar Rust**. The model achieved approximately **80.82% test accuracy** on the unseen test dataset.

The experiment also demonstrated that data augmentation can help generalization, while the confusion between scab and black rot shows the limitations caused by the small dataset and visually similar symptoms.
